# 🖐 Control de Volumen con Landmarks de Mano

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

Este laboratorio demuestra el uso de **MediaPipe Hand Landmarker** para detectar puntos clave de la mano en tiempo real y traducir esa información gestual en una acción concreta: **subir y bajar el volumen del sistema**.

---

## ✦ Concepto

La mano tiene **21 landmarks** numerados. Vamos a usar la **posición vertical (eje Y) de la muñeca** como control:

- Mano **arriba** → volumen **alto**
- Mano **abajo** → volumen **bajo**

```
  4
  |   8  12  16  20
  3   |   |   |   |
  |   7  11  15  19
  2   |   |   |   |
  |   6  10  14  18
  1   |   |   |   |
   \  5   9  13  17
    \ |   |   |   |
     [0: MUÑECA]      ← usamos este punto
```

El landmark `0` (muñeca) tiene coordenada `y` normalizada entre `0.0` (arriba de la imagen) y `1.0` (abajo). Invertimos esa escala para mapearla a volumen (0 %–100 %).

## Objetivo

El objetivo de este laboratorio es construir una aplicación de visión artificial en tiempo real que detecte la posición de la mano mediante **MediaPipe Hand Landmarker** y traduzca esa información gestual en una acción concreta del sistema operativo: el control del volumen de audio.

## Al terminar este material vas a poder:

1. Descargar e instanciar un modelo de detección de puntos clave (*landmarks*) de MediaPipe.
2. Implementar una función de mapeo entre una coordenada normalizada y un rango de valores de salida.
3. Desarrollar un loop de cámara en tiempo real con OpenCV.
4. Integrar la salida de un modelo de visión artificial con una acción del sistema operativo.

## Terminología clave (Microglosario)

*   **Landmark (Punto Clave):** Coordenada geométrica de referencia que el modelo aprende a localizar en la imagen. *Como las articulaciones y nudillos que reconocerías en la radiografía de una mano: hay 21 puntos definidos que el modelo aprende a ubicar con precisión en cada cuadro.*
*   **Coordenada normalizada:** Valor entre 0.0 y 1.0 que representa una posición relativa dentro de la imagen, independientemente de su resolución. *Como decir "el objeto está al 30 % del ancho de la pantalla" en lugar de "en el píxel 384": funciona igual en cualquier tamaño de ventana.*
*   **Suavizado exponencial:** Técnica que promedia gradualmente el valor nuevo con el anterior para eliminar saltos bruscos. *Como el comportamiento del termostato de un auto: no reacciona de golpe a cada cambio de temperatura, sino que avanza suavemente hacia el nuevo valor.*
*   **HUD (Heads-Up Display):** Superposición gráfica de información de estado sobre la imagen en tiempo real. *Como el velocímetro proyectado en el parabrisas de un avión: la información aparece encima de la escena sin interrumpirla.*
*   **Hand Landmarker:** Modelo preentrenado de MediaPipe que localiza los 21 puntos clave de la mano en cada cuadro de video. *Como un GPS que, en lugar de ciudades, mapea en tiempo real las articulaciones de los dedos.*

## Paso 1 — Preparar el entorno y el modelo

Instalá las dependencias desde la terminal con `uv sync` (o `uv add ...`) y luego ejecutá esta celda **una sola vez** para descargar el modelo pre-entrenado.

In [ ]:
# Descarga del modelo Hand Landmarker
import urllib.request, os

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✓ Modelo descargado:", MODEL_PATH)
else:
    print("✓ Modelo ya disponible:", MODEL_PATH)

## Paso 2 — Funciones auxiliares

Definimos las funciones para:
1. Detectar la mano
2. Mapear la posición Y al rango de volumen
3. Ejecutar el cambio de volumen en el sistema operativo, incluido Windows con `pycaw`

In [ ]:
import mediapipe as mp
import cv2
import subprocess
import platform

# --- Configuración del Hand Landmarker ---

OpcionesBase          = mp.tasks.BaseOptions
DetectorManos         = mp.tasks.vision.HandLandmarker
OpcionesDetectorManos = mp.tasks.vision.HandLandmarkerOptions
ModoEjecucion         = mp.tasks.vision.RunningMode

opciones_landmarker = OpcionesDetectorManos(
    base_options=OpcionesBase(model_asset_path='hand_landmarker.task'),
    running_mode=ModoEjecucion.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)


# --- Control de volumen según el sistema operativo ---

SISTEMA_OPERATIVO = platform.system()  # 'Darwin' = macOS, 'Linux', 'Windows'
print(f"✓ Sistema operativo detectado: {SISTEMA_OPERATIVO}")

VOLUMEN_WINDOWS = None
ADVERTENCIA_VOLUMEN_MOSTRADA = False

if SISTEMA_OPERATIVO == 'Windows':
    try:
        from comtypes import CoInitialize
        from pycaw.pycaw import AudioUtilities

        CoInitialize()
        dispositivos = AudioUtilities.GetSpeakers()
        VOLUMEN_WINDOWS = dispositivos.EndpointVolume
        print("✓ Control de volumen de Windows inicializado.")
    except Exception as e:
        VOLUMEN_WINDOWS = None
        print(f"Aviso: no se pudo inicializar pycaw ({e})")


def ajustar_volumen(nivel: int):
    """Establece el volumen del sistema operativo. nivel: 0-100"""
    global ADVERTENCIA_VOLUMEN_MOSTRADA
    nivel = max(0, min(100, nivel))
    try:
        if SISTEMA_OPERATIVO == 'Darwin':
            subprocess.run(
                ['osascript', '-e', f'set volume output volume {nivel}'],
                capture_output=True
            )
        elif SISTEMA_OPERATIVO == 'Linux':
            subprocess.run(
                ['amixer', '-q', 'sset', 'Master', f'{nivel}%'],
                capture_output=True
            )
        elif SISTEMA_OPERATIVO == 'Windows':
            if VOLUMEN_WINDOWS is None:
                return
            VOLUMEN_WINDOWS.SetMute(0, None)
            VOLUMEN_WINDOWS.SetMasterVolumeLevelScalar(nivel / 100.0, None)
    except Exception as e:
        if not ADVERTENCIA_VOLUMEN_MOSTRADA:
            print(f"Aviso: no se pudo aplicar el volumen del sistema ({e})")
            ADVERTENCIA_VOLUMEN_MOSTRADA = True


# --- Mapeo de coordenada Y normalizada a porcentaje de volumen ---
# y=0.0 es la parte superior de la imagen → volumen alto
# y=1.0 es la parte inferior               → volumen bajo

def y_a_volumen(coordenada_y: float) -> int:
    """Convierte coordenada y normalizada [0,1] a porcentaje de volumen [0,100]."""
    volumen = int((1.0 - coordenada_y) * 100)
    return max(0, min(100, volumen))


# --- Visualización: dibuja landmarks y HUD sobre el cuadro ---

COLOR_LANDMARK = (0, 255, 120)    # verde para los puntos clave
COLOR_CONEXION = (255, 255, 255)  # blanco para las líneas
COLOR_MUNECA   = (0, 120, 255)    # naranja para destacar la muñeca

# Conexiones estándar de MediaPipe (pares de índices de landmarks)
CONEXIONES_MANO = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]


def dibujar_landmarks(cuadro, puntos_clave):
    alto, ancho = cuadro.shape[:2]

    # Convertimos cada coordenada normalizada a píxeles
    puntos = []
    for lm in puntos_clave:
        x_pixel = int(lm.x * ancho)
        y_pixel = int(lm.y * alto)
        puntos.append((x_pixel, y_pixel))

    # Dibujamos las líneas de conexión entre landmarks
    for conexion in CONEXIONES_MANO:
        punto_inicio = conexion[0]
        punto_fin    = conexion[1]
        cv2.line(cuadro, puntos[punto_inicio], puntos[punto_fin],
                 COLOR_CONEXION, 1, cv2.LINE_AA)

    # Dibujamos cada punto clave (la muñeca destacada en color diferente)
    for i in range(len(puntos)):
        px, py = puntos[i]
        color_punto = COLOR_MUNECA if i == 0 else COLOR_LANDMARK
        radio = 7 if i == 0 else 4
        cv2.circle(cuadro, (px, py), radio, color_punto, -1, cv2.LINE_AA)

    return puntos


def dibujar_hud(cuadro, volumen: int, mano_detectada: bool):
    alto, ancho = cuadro.shape[:2]

    # Barra de volumen en el lateral derecho
    barra_x, barra_y        = ancho - 50, 30
    barra_alto, barra_ancho = alto - 60, 30

    # Fondo gris de la barra
    cv2.rectangle(cuadro,
                  (barra_x, barra_y),
                  (barra_x + barra_ancho, barra_y + barra_alto),
                  (60, 60, 60), -1)

    # Relleno proporcional al nivel de volumen actual
    relleno_alto  = int(barra_alto * volumen / 100)
    color_relleno = (0, 200 + int(55 * volumen / 100), 100)
    cv2.rectangle(cuadro,
                  (barra_x, barra_y + barra_alto - relleno_alto),
                  (barra_x + barra_ancho, barra_y + barra_alto),
                  color_relleno, -1)

    # Borde de la barra
    cv2.rectangle(cuadro,
                  (barra_x, barra_y),
                  (barra_x + barra_ancho, barra_y + barra_alto),
                  (180, 180, 180), 1)

    # Etiqueta con el porcentaje
    cv2.putText(cuadro, f"{volumen}%",
                (barra_x - 5, barra_y + barra_alto + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "VOL",
                (barra_x + 2, barra_y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA)

    # Estado de detección de mano
    estado       = "MANO DETECTADA" if mano_detectada else "Sin detección"
    color_estado = (0, 255, 120) if mano_detectada else (80, 80, 80)
    cv2.putText(cuadro, estado, (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color_estado, 1, cv2.LINE_AA)

    # Instrucciones en pantalla
    cv2.putText(cuadro, "Mano arriba = volumen alto", (15, alto - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "Mano abajo  = volumen bajo", (15, alto - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "[Q] para salir", (15, alto - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)


print("✓ Funciones auxiliares cargadas.")

## Paso 3 — Loop principal

Ejecutá esta celda para iniciar la webcam. Una ventana de OpenCV se abrirá con la cámara en tiempo real.

- **Mové la mano** dentro del cuadro
- La barra lateral muestra el volumen en tiempo real
- **Presioná `Q`** para cerrar

> ℹ️ Si la cámara no abre o el volumen no cambia, cerrá otras apps que estén usando la webcam o el audio. En macOS, aceptá los permisos de cámara si el sistema los solicita.

In [ ]:
# ─── LOOP PRINCIPAL ────────────────────────────────────────────────────────────

captura = cv2.VideoCapture(0)   # 0 = cámara por defecto del sistema

# Suavizado exponencial del volumen para evitar saltos bruscos
SUAVIZADO         = 0.15   # entre 0 (muy lento) y 1 (sin suavizado)
volumen_suavizado = 50     # volumen inicial en porcentaje

# Limitamos la frecuencia de llamadas al SO para no sobrecargarlo
APLICAR_CADA_N_CUADROS = 5
contador_cuadros        = 0
ultimo_volumen_aplicado = None

print("✦ Cámara activa. Presioná Q en la ventana para cerrar.")

try:
    if not captura.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara. Verificá los permisos.")

    with DetectorManos.create_from_options(opciones_landmarker) as landmarker:
        while True:
            lectura_exitosa, cuadro = captura.read()
            if not lectura_exitosa:
                break

            # Espejamos horizontalmente para una interacción más intuitiva
            cuadro = cv2.flip(cuadro, 1)

            # Convertimos el cuadro a RGB (MediaPipe requiere este formato)
            cuadro_rgb = cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)
            imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=cuadro_rgb)

            # Ejecutamos la detección de puntos clave
            deteccion = landmarker.detect(imagen_mp)

            mano_detectada   = len(deteccion.hand_landmarks) > 0
            volumen_objetivo = volumen_suavizado  # si no hay mano, mantenemos el actual

            if mano_detectada:
                puntos_clave      = deteccion.hand_landmarks[0]  # primera mano detectada
                posicion_muneca_y = puntos_clave[0].y             # landmark 0 = muñeca

                volumen_objetivo = y_a_volumen(posicion_muneca_y)
                dibujar_landmarks(cuadro, puntos_clave)

            # Suavizado exponencial: avanzamos lentamente hacia el volumen objetivo
            volumen_suavizado = volumen_suavizado + SUAVIZADO * (volumen_objetivo - volumen_suavizado)
            volumen_mostrado  = int(volumen_suavizado)

            # Aplicamos el volumen al SO cada N cuadros para no saturarlo
            contador_cuadros += 1
            if contador_cuadros % APLICAR_CADA_N_CUADROS == 0 and volumen_mostrado != ultimo_volumen_aplicado:
                ajustar_volumen(volumen_mostrado)
                ultimo_volumen_aplicado = volumen_mostrado

            # Superponemos el HUD y mostramos el cuadro
            dibujar_hud(cuadro, volumen_mostrado, mano_detectada)
            cv2.imshow('Control de Volumen — MediaPipe Hands', cuadro)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("✦ Saliendo...")
                break

finally:
    captura.release()
    cv2.destroyAllWindows()
    print("✓ Sesión cerrada.")

---

## ✎ Para pensar antes de continuar

Antes de pasar a las variaciones, respondé estas preguntas mirando el código que acabás de ejecutar:

1. ¿Qué pasaría si cambiás `SUAVIZADO` a `1.0`? ¿Y a `0.01`? ¿Qué trade-off controlás con ese parámetro?
2. ¿Por qué se invierte la coordenada Y en `y_a_volumen`? Describí con tus palabras cómo funciona el sistema de coordenadas de MediaPipe.
3. ¿Por qué el volumen se aplica cada `APLICAR_CADA_N_CUADROS` cuadros en lugar de en cada uno? ¿Qué consecuencias tendría hacerlo en cada frame?

---

## Para explorar en clase

Una vez que el ejemplo base funciona, estas variaciones son buenos ejercicios para extender el laboratorio:

### Variación 1 — Usar la distancia entre dedos
En lugar de la posición de la muñeca, usá la **distancia entre el pulgar (landmark 4) y el índice (landmark 8)** como control. Es el gesto clásico de "pinch-to-zoom".

```python
import math

def distancia(lm_a, lm_b):
    return math.sqrt((lm_a.x - lm_b.x)**2 + (lm_a.y - lm_b.y)**2)

d = distancia(puntos_clave[4], puntos_clave[8])  # pulgar - índice
# d ≈ 0.0 (dedos juntos) a 0.4 (dedos separados)
volumen_objetivo = int((d / 0.4) * 100)
```

### Variación 2 — Usar ambas manos
Cambiá `num_hands=2` en las opciones y usá una mano para el volumen y la otra para el brillo.

### Variación 3 — Contar dedos levantados
Detectá si cada dedo está extendido (comparando la punta con la articulación media) y mapeá el conteo (0–5) a valores discretos de volumen: 0 %, 20 %, 40 %, 60 %, 80 %, 100 %.

---

## Referencias

- [MediaPipe Hand Landmarker — Documentación oficial](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker)
- [Índice de landmarks de la mano](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker#hand_landmark_model_bundle)
- [pycaw — Control de volumen en Windows](https://github.com/AndreMiras/pycaw)